<a href="https://colab.research.google.com/github/Sai7236/phishing-detection-system/blob/main/Email_threat_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
import os, torch, joblib
drive.mount('/content/drive', force_remount=True)
BASE   = '/content/drive/MyDrive/Phishing_Project'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('✅ BASE =', BASE, '| device =', device)

Mounted at /content/drive
✅ BASE = /content/drive/MyDrive/Phishing_Project | device = cpu


In [ ]:
!pip install -q transformers datasets accelerate torch
!pip install -q scikit-learn shap lime tldextract joblib
!pip install -q git+https://github.com/openai/CLIP.git
!pip install -q gradio mlflow fastapi uvicorn nest-asyncio pyngrok
print('✅ Libraries installed')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 5.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 7.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.4/49.4 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 69.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 81.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import clip

lexical_clf    = joblib.load(f'{BASE}/Models/lexical_rf.pkl')
tokenizer_ling = AutoTokenizer.from_pretrained(f'{BASE}/Models/ling_model')
model_ling     = AutoModelForSequenceClassification.from_pretrained(
                     f'{BASE}/Models/ling_model').to(device)
model_ling.eval()
clip_model, clip_prep = clip.load('ViT-B/32', device=device)
clip_model.eval()
print('✅ All 3 models loaded')

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

100%|████████████████████████████████████████| 338M/338M [00:02<00:00, 139MiB/s]


✅ All 3 models loaded


In [ ]:
# ══════════════════════════════════════════════════════════════════
# ADDITIONAL DATA — Government & Legitimate Site Coverage
# Run this ONCE after Phase 1 to augment your URL dataset
# ══════════════════════════════════════════════════════════════════
import pandas as pd, os
BASE = "/content/drive/MyDrive/Phishing_Project"

# --- Trusted TLD patterns for Indian + Global government sites ---
TRUSTED_TLDS = {
    # Indian government
    ".gov.in", ".nic.in", ".ac.in", ".edu.in", ".res.in",
    ".mil.in", ".net.in", ".org.in",
    # Global government
    ".gov", ".gov.uk", ".gov.au", ".gov.ca", ".gouv.fr",
    ".govt.nz", ".gov.sg", ".gov.za", ".gob.mx", ".gov.br",
    # Education & International Orgs
    ".edu", ".ac.uk", ".ac.in", ".un.org", ".who.int",
}

# --- Synthetic safe URL list for Indian government portals ---
# These are real domains — adding them as label=0 (safe)
indian_gov_urls = [
    "https://www.india.gov.in",
    "https://incometaxindia.gov.in",
    "https://www.irctc.co.in",
    "https://www.sbi.co.in",
    "https://epfindia.gov.in",
    "https://www.mca.gov.in",
    "https://digitalindia.gov.in",
    "https://www.nhp.gov.in",
    "https://www.meity.gov.in",
    "https://uidai.gov.in",
    "https://www.niti.gov.in",
    "https://pib.gov.in",
    "https://www.isro.gov.in",
    "https://www.drdo.gov.in",
    "https://iitb.ac.in",
    "https://iitd.ac.in",
    "https://www.aiims.edu",
    "https://www.iisc.ac.in",
    "https://ugc.ac.in",
    "https://naac.gov.in",
    "https://nta.ac.in",
    "https://cbse.gov.in",
    "https://www.rbi.org.in",
    "https://sebi.gov.in",
    "https://www.irdai.gov.in",
    "https://cowin.gov.in",
    "https://diksha.gov.in",
    "https://www.passport.gov.in",
    "https://www.eci.gov.in",
    "https://vikaspedia.in",
    # International government
    "https://www.usa.gov",
    "https://www.gov.uk",
    "https://www.canada.ca",
    "https://www.australia.gov.au",
    "https://www.un.org",
    "https://www.who.int",
    "https://www.worldbank.org",
    "https://www.imf.org",
]

gov_df = pd.DataFrame({"url": indian_gov_urls, "label": 0})

# Load existing URL dataset and append
url_df = pd.read_csv(f"{BASE}/Data/malicious_phish.csv")
url_df["label"] = url_df["type"].map({
    "benign":0,"defacement":1,"phishing":1,"malware":1
})
url_df = url_df[["url","label"]].dropna()

# Append government URLs (repeat 50x to give them enough weight)
gov_augmented = pd.concat([gov_df]*50, ignore_index=True)
url_df_augmented = pd.concat([url_df, gov_augmented], ignore_index=True)

save_path = f"{BASE}/Data/malicious_phish_augmented.csv"
url_df_augmented.to_csv(save_path, index=False)
print(f"✅ Augmented dataset saved: {len(url_df_augmented)} rows")
print(f"   New safe entries added: {len(gov_augmented)}")

✅ Augmented dataset saved: 653091 rows
   New safe entries added: 1900


In [ ]:
# ══════════════════════════════════════════════════════════════════
# HARDENED TRUST CHECK — handles gov.in, nic.in, ac.in etc.
# Replace your existing is_trusted_domain() with this
# ══════════════════════════════════════════════════════════════════
import tldextract, re

# Exact domain whitelist
EXACT_TRUSTED = {
    "google.com","gmail.com","youtube.com","microsoft.com","github.com",
    "amazon.com","apple.com","netflix.com","linkedin.com","wikipedia.org",
    "outlook.com","office.com","live.com","bing.com","yahoo.com",
    "twitter.com","facebook.com","instagram.com","whatsapp.com",
    # Indian trusted
    "irctc.co.in","sbi.co.in","rbi.org.in","uidai.gov.in","npci.org.in",
}

# Trusted TLD suffixes — any domain ending in these is treated as safe
TRUSTED_TLD_SUFFIXES = (
    ".gov.in",".nic.in",".ac.in",".edu.in",".res.in",".mil.in",
    ".gov.uk",".gov.au",".gov.ca",".gouv.fr",".govt.nz",
    ".gov",".edu",".ac.uk",".un.org",".who.int",".int",
)

# Trusted second-level domains (any subdomain of these is safe)
TRUSTED_SLD_PATTERNS = [
    r".*\.gov\.in$", r".*\.nic\.in$", r".*\.ac\.in$", r".*\.edu\.in$",
    r".*\.gov\.uk$", r".*\.gov\.au$", r".*\.gc\.ca$",
    r".*\.gov$",     r".*\.edu$",     r".*\.mil$",
]

def is_trusted_domain(url: str) -> tuple:
    """
    Returns (is_trusted: bool, reason: str)
    Handles Indian government, international government, and education domains.
    """
    try:
        url_lower = url.lower().strip()
        ext = tldextract.extract(url_lower)
        registered = f"{ext.domain}.{ext.suffix}".lower()
        full_host  = f"{ext.subdomain}.{ext.domain}.{ext.suffix}".lower().lstrip(".")

        # 1. Exact match
        if registered in EXACT_TRUSTED:
            return True, f"Exact whitelist match: {registered}"

        # 2. TLD suffix match
        for suffix in TRUSTED_TLD_SUFFIXES:
            if full_host.endswith(suffix) or registered.endswith(suffix):
                return True, f"Trusted TLD: {suffix}"

        # 3. Regex pattern match
        for pattern in TRUSTED_SLD_PATTERNS:
            if re.match(pattern, full_host) or re.match(pattern, registered):
                return True, f"Trusted domain pattern: {pattern}"

        return False, ""

    except Exception:
        return False, ""

# Test it
tests = [
    "https://incometaxindia.gov.in/pages/home.aspx",
    "https://iitb.ac.in/admissions",
    "https://www.rbi.org.in",
    "https://www.gov.uk/browse/benefits",
    "http://paypa1-secure.xyz/login",
    "https://netflix-billing.net/update",
]
for t in tests:
    trusted, reason = is_trusted_domain(t)
    status = "✅ TRUSTED" if trusted else "🔍 ANALYSE"
    print(f"{status} — {t[:55]:<55} {reason}")

✅ TRUSTED — https://incometaxindia.gov.in/pages/home.aspx           Trusted TLD: .gov.in
✅ TRUSTED — https://iitb.ac.in/admissions                           Trusted TLD: .ac.in
✅ TRUSTED — https://www.rbi.org.in                                  Exact whitelist match: rbi.org.in
✅ TRUSTED — https://www.gov.uk/browse/benefits                      Trusted TLD: .gov.uk
🔍 ANALYSE — http://paypa1-secure.xyz/login                          
🔍 ANALYSE — https://netflix-billing.net/update                      


In [ ]:
# ══════════════════════════════════════════════════════════════════
# PREMIUM GRADIO FRONTEND — Full upgrade with custom CSS
# Paste this as your final Gradio cell in Colab
# ══════════════════════════════════════════════════════════════════
import gradio as gr, torch, clip, shap, joblib, re, math
import numpy as np, pandas as pd, transformers
from PIL import Image
from urllib.parse import urlparse
import tldextract, lime.lime_tabular as lt
from transformers import AutoTokenizer, AutoModelForSequenceClassification

BASE   = "/content/drive/MyDrive/Phishing_Project"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

lexical_clf    = joblib.load(f"{BASE}/Models/lexical_rf.pkl")
tokenizer_ling = AutoTokenizer.from_pretrained(f"{BASE}/Models/ling_model")
model_ling     = AutoModelForSequenceClassification.from_pretrained(
                     f"{BASE}/Models/ling_model").to(device)
model_ling.eval()
clip_model, clip_prep = clip.load("ViT-B/32", device=device)
clip_model.eval()

FEATURE_ORDER = ["url_length","hostname_length","num_dots","has_at_symbol","has_ip",
                 "subdomain_depth","entropy","is_https","path_length","num_hyphens",
                 "num_slashes","num_query_params","has_port","tld_suspicious",
                 "suspicious_words","digit_ratio_host","url_has_redirect","double_slash_path"]

# ── Expanded trusted domain check (fixes gov.in, nic.in, ac.in) ──
EXACT_TRUSTED = {
    "google.com","gmail.com","youtube.com","microsoft.com","github.com",
    "amazon.com","apple.com","netflix.com","linkedin.com","wikipedia.org",
    "outlook.com","office.com","irctc.co.in","sbi.co.in","rbi.org.in",
}
TRUSTED_TLD_SUFFIXES = (
    ".gov.in",".nic.in",".ac.in",".edu.in",".res.in",
    ".gov.uk",".gov.au",".gov.ca",".gouv.fr",".govt.nz",
    ".gov",".edu",".ac.uk",".int",".mil",
)

def is_trusted(url):
    try:
        ext  = tldextract.extract(url.lower())
        reg  = f"{ext.domain}.{ext.suffix}"
        full = f"{ext.subdomain}.{ext.domain}.{ext.suffix}".lstrip(".")
        if reg in EXACT_TRUSTED: return True, f"Whitelisted: {reg}"
        for s in TRUSTED_TLD_SUFFIXES:
            if full.endswith(s) or reg.endswith(s):
                return True, f"Trusted TLD: {s}"
        return False, ""
    except: return False, ""

def extract_features(url_str):
    try:
        parsed=urlparse(url_str); ext=tldextract.extract(url_str)
        host=parsed.hostname or ""; path=parsed.path or ""; query=parsed.query or ""
        chars=set(url_str)
        entropy=-sum((url_str.count(c)/len(url_str))*math.log2(url_str.count(c)/len(url_str))
                     for c in chars if url_str.count(c)>0) if url_str else 0
        sw=["login","verify","secure","update","account","banking","confirm",
            "password","signin","paypal","ebay","amazon","apple","microsoft"]
        dr=sum(c.isdigit() for c in host)/len(host) if host else 0
        return {"url_length":len(url_str),"hostname_length":len(host),"num_dots":url_str.count("."),
                "has_at_symbol":int("@" in url_str),"has_ip":int(bool(re.match(r"^\d{1,3}(\.\d{1,3}){3}$",host))),
                "subdomain_depth":len(ext.subdomain.split(".")) if ext.subdomain else 0,
                "entropy":round(entropy,4),"is_https":int(parsed.scheme=="https"),
                "path_length":len(path),"num_hyphens":url_str.count("-"),
                "num_slashes":url_str.count("/"),"num_query_params":len(query.split("&")) if query else 0,
                "has_port":int(bool(parsed.port)),"tld_suspicious":int(ext.suffix in ["xyz","tk","ml","ga","cf","top","click"]),
                "suspicious_words":sum(1 for w in sw if w in url_str.lower()),
                "digit_ratio_host":round(dr,4),
                "url_has_redirect":int("redirect" in url_str.lower() or "url=" in url_str.lower()),
                "double_slash_path":int("//" in path)}
    except: return {k:0 for k in FEATURE_ORDER}

def get_shap(text):
    try:
        pipe=transformers.pipeline("text-classification",model=model_ling,
                                   tokenizer=tokenizer_ling,return_all_scores=True,
                                   device=0 if torch.cuda.is_available() else -1)
        sv=shap.Explainer(pipe)([text])
        pc=int(np.argmax([sv.values[0,:,i].sum() for i in range(3)]))
        pairs=list(zip(sv.data[0],sv.values[0,:,pc].tolist()))
        return sorted(pairs,key=lambda x:abs(x[1]),reverse=True)[:5]
    except: return []

_lime_exp=None
def get_lime(url_str):
    global _lime_exp
    try:
        if _lime_exp is None:
            _lime_exp=lt.LimeTabularExplainer(
                np.zeros((10,len(FEATURE_ORDER))),
                feature_names=FEATURE_ORDER,class_names=["Safe","Malicious"],mode="classification")
        feats=np.array(list(extract_features(url_str).values()))
        exp=_lime_exp.explain_instance(feats,lexical_clf.predict_proba,num_features=4)
        return [(f,s) for f,s in exp.as_list() if s>0.01]
    except: return []

def check_logo(image, brand):
    if image is None or not brand.strip():
        return None, "No image provided or brand name is empty"
    try:
        # 1. FIX FORMAT: Gradio passes PIL images if type="pil", but let's force RGB safety
        if not isinstance(image, Image.Image):
            image = Image.fromarray(image)

        # 2. Preprocess the image for the ViT-B/32 backbone
        t = clip_prep(image.convert("RGB")).unsqueeze(0).to(device)

        # 3. Use strict context-anchored prompts to maximize accuracy
        p = clip.tokenize([
            f"a close up photo of the official {brand} logo",
            "a generic unrelated website asset or random background image"
        ]).to(device)

        with torch.no_grad():
            # 4. Extract features from both modalities
            i = clip_model.encode_image(t)
            tx = clip_model.encode_text(p)

            # CRITICAL CORRECTION: You MUST perform L2 normalization
            # otherwise the cosine similarity calculations will be completely chaotic
            i = i / i.norm(dim=-1, keepdim=True)
            tx = tx / tx.norm(dim=-1, keepdim=True)

            # 5. Apply the standard CLIP temperature scaling matrix (100.0) and Softmax
            sim = (100.0 * i @ tx.T).softmax(dim=-1).squeeze()

        m = sim[0].item() # The probability that it matches the claimed brand logo

        # Invert it for the threat pipeline: High match means LOW visual threat score
        visual_threat_score = 1.0 - m

        status = f"Brand match confidence: {m*100:.1f}% — " + \
                 ("🚨 IMPERSONATION DETECTED" if m < 0.55 else "✅ Verified Official Logo")

        return visual_threat_score, status

    except Exception as e:
        return None, f"CLIP Error: {str(e)}"

# ── Core scan logic ───────────────────────────────────────────────
def scan(email_text, url_input, image, brand_name, spf_fail, dkim_fail):
    s_ling=s_lex=s_vis=None
    evidence=[]; flags=[]

    if email_text and email_text.strip():
        inputs=tokenizer_ling(email_text,return_tensors="pt",truncation=True,max_length=256).to(device)
        with torch.no_grad():
            probs=torch.softmax(model_ling(**inputs).logits,dim=-1)[0].tolist()
        s_ling=min(probs[2]+0.5*probs[1],1.0)
        for tok,sc in get_shap(email_text):
            if abs(sc)>0.03: evidence.append(f"Suspicious token '{tok}' — SHAP impact {sc:+.3f}")

    if url_input and url_input.strip():
        url_str=url_input.strip()
        if not url_str.startswith(("http://","https://")): url_str="http://"+url_str
        trusted,reason=is_trusted(url_str)
        if trusted:
            flags.append("TRUSTED_DOMAIN")
            s_lex=0.0
            evidence.append(f"Domain whitelisted — {reason}")
        else:
            feats=np.array(list(extract_features(url_str).values()))
            s_lex=float(lexical_clf.predict_proba([feats])[0][1])
            for feat,sc in get_lime(url_str):
                evidence.append(f"URL pattern: {feat} — LIME score {sc:+.3f}")

    if image is not None and brand_name and brand_name.strip():
        s_vis,clip_detail=check_logo(image,brand_name)
        evidence.append(f"Visual — {clip_detail}")

    if spf_fail:  flags.append("SPF_FAIL");  evidence.append("SPF authentication failed")
    if dkim_fail: flags.append("DKIM_FAIL"); evidence.append("DKIM authentication failed")

    # Fusion
    if spf_fail and dkim_fail:
        verdict,score,conf="PHISHING",1.0,"High (hard override)"
    elif s_vis is not None and s_vis>0.85:
        verdict,score,conf="PHISHING",1.0,"High (visual override)"
    else:
        W={"ling":0.40,"lex":0.35,"vis":0.25}
        sc_map={k:v for k,v in [("ling",s_ling),("lex",s_lex),("vis",s_vis)] if v is not None}
        if not sc_map: return "No input","—","—","—","—","Please enter an email or URL."
        tw=sum(W[k] for k in sc_map)
        score=sum(sc_map[k]*W[k]/tw for k in sc_map)
        verdict="PHISHING" if score>=0.65 else ("SCAM" if score>=0.35 else "GENUINE")
        conf="High" if score>=0.80 else ("Medium" if score>=0.50 else "Low")

    ling_str=f"{s_ling*100:.1f}%" if s_ling is not None else "Not used"
    lex_str =f"{s_lex*100:.1f}%"  if s_lex  is not None else "Not used"
    vis_str =f"{s_vis*100:.1f}%"  if s_vis  is not None else "Not used"
    flags_str=", ".join(flags) if flags else "None"
    ev_str="\n".join(f"  • {e}" for e in evidence[:6]) if evidence else "  No specific indicators"
    report=f"Flags: {flags_str}\n\nEvidence (XAI):\n{ev_str}"

    return verdict, f"{score*100:.1f}%", conf, ling_str, lex_str, vis_str, report

# ── Custom CSS for premium look ───────────────────────────────────
css = """
body { font-family: 'Inter', sans-serif !important; }
.gradio-container { max-width: 900px !important; margin: auto; }

/* Header bar */
.header-bar {
    background: linear-gradient(135deg, #0D1B2A 0%, #1B3A5C 100%);
    border-radius: 12px; padding: 20px 24px; margin-bottom: 16px;
    display: flex; align-items: center; gap: 16px;
}
.header-title { color: #fff; font-size: 22px; font-weight: 600; }
.header-sub   { color: #8FA3B1; font-size: 13px; margin-top: 2px; }
.badge        { background: #02C39A22; color: #02C39A; border: 1px solid #02C39A55;
                border-radius: 20px; padding: 3px 12px; font-size: 12px; font-weight: 500; }

/* Verdict output */
.verdict-phishing textarea { background: #FFF0F0 !important; border: 2px solid #E24B4A !important;
    font-size: 20px !important; font-weight: 700 !important; color: #A32D2D !important; }
.verdict-genuine textarea  { background: #F0FFF4 !important; border: 2px solid #2DC653 !important;
    font-size: 20px !important; font-weight: 700 !important; color: #1a7a30 !important; }
.verdict-scam textarea     { background: #FFFBF0 !important; border: 2px solid #F4A261 !important;
    font-size: 20px !important; font-weight: 700 !important; color: #854F0B !important; }

/* Score cards */
.score-card textarea { background: #F8FAFC !important; text-align: center !important;
    font-size: 18px !important; font-weight: 600 !important; }

/* Report box */
.report-box textarea { background: #0D1B2A !important; color: #9FE1CB !important;
    font-family: 'Courier New', monospace !important; font-size: 13px !important;
    border: 1px solid #1B3A5C !important; }

/* Scan button */
#scan-btn { background: linear-gradient(135deg, #028090, #02C39A) !important;
    border: none !important; color: white !important; font-size: 16px !important;
    font-weight: 600 !important; padding: 14px !important; border-radius: 10px !important; }
#scan-btn:hover { opacity: 0.9 !important; transform: translateY(-1px); }

/* Tab styling */
.tab-nav button { font-weight: 500 !important; }

/* Input areas */
.input-card { border: 1px solid #E2E8F0; border-radius: 10px; padding: 4px; }
"""

# ── Gradio UI ─────────────────────────────────────────────────────
with gr.Blocks(css=css, title="PhishGuard AI") as demo:

    gr.HTML("""
    <div class="header-bar">
      <svg width="40" height="40" viewBox="0 0 32 32" fill="none">
        <path d="M16 2L4 7v9c0 7.18 5.16 13.9 12 15.5C22.84 29.9 28 23.18 28 16V7L16 2z"
              fill="#028090" opacity=".3"/>
        <path d="M16 2L4 7v9c0 7.18 5.16 13.9 12 15.5C22.84 29.9 28 23.18 28 16V7L16 2z"
              stroke="#02C39A" stroke-width="1.5" fill="none"/>
        <path d="M11 16l3 3 7-7" stroke="#02C39A" stroke-width="2.5"
              stroke-linecap="round" stroke-linejoin="round"/>
      </svg>
      <div style="flex:1">
        <div class="header-title">PhishGuard AI</div>
      </div>
    </div>
    """)

    with gr.Tabs():

        with gr.TabItem("Full scan"):
            with gr.Row():
                with gr.Column(scale=3):
                    email_in = gr.Textbox(
                        label="Email body",
                        placeholder="Paste full email text here...",
                        lines=5, elem_classes=["input-card"]
                    )
                    url_in = gr.Textbox(
                        label="URL to check",
                        placeholder="e.g. http://paypa1-secure.xyz/login",
                        elem_classes=["input-card"]
                    )
                    with gr.Row():
                        spf_cb  = gr.Checkbox(label="SPF failure detected",  value=False)
                        dkim_cb = gr.Checkbox(label="DKIM failure detected", value=False)

                with gr.Column(scale=2):
                    image_in = gr.Image(label="Logo / screenshot (for CLIP)", type="pil", height=160)
                    brand_in = gr.Textbox(label="Claimed brand name", placeholder="e.g. PayPal")

            scan_btn = gr.Button("Run full threat scan", elem_id="scan-btn", size="lg")

            gr.Markdown("### Results")
            with gr.Row():
                verdict_out = gr.Textbox(label="Verdict",    lines=1, elem_classes=["verdict-phishing"])
                score_out   = gr.Textbox(label="Threat score", lines=1, elem_classes=["score-card"])
                conf_out    = gr.Textbox(label="Confidence",  lines=1, elem_classes=["score-card"])

            with gr.Row():
                ling_out = gr.Textbox(label="Linguistic engine", lines=1, elem_classes=["score-card"])
                lex_out  = gr.Textbox(label="Lexical engine",    lines=1, elem_classes=["score-card"])
                vis_out  = gr.Textbox(label="Visual engine",     lines=1, elem_classes=["score-card"])

            report_out = gr.Textbox(
                label="XAI evidence report",
                lines=8, elem_classes=["report-box"]
            )

            scan_btn.click(
                fn=scan,
                inputs=[email_in, url_in, image_in, brand_in, spf_cb, dkim_cb],
                outputs=[verdict_out, score_out, conf_out, ling_out, lex_out, vis_out, report_out]
            )

        with gr.TabItem("Quick URL check"):
            gr.Markdown("### Rapid URL analysis — Lexical engine only")
            q_url = gr.Textbox(label="URL", placeholder="http://example.com/login")
            q_btn = gr.Button("Check URL", variant="primary")
            with gr.Row():
                q_verdict = gr.Textbox(label="Verdict")
                q_score   = gr.Textbox(label="Score")
            def quick_url(url):
                if not url.strip(): return "Enter a URL", "—"
                if not url.startswith(("http://","https://")): url="http://"+url
                trusted,reason=is_trusted(url)
                if trusted: return f"GENUINE — {reason}", "0.0%"
                feats=np.array(list(extract_features(url).values()))
                sc=float(lexical_clf.predict_proba([feats])[0][1])
                v="PHISHING" if sc>=0.65 else ("SCAM" if sc>=0.35 else "GENUINE")
                return v, f"{sc*100:.1f}%"
            q_btn.click(fn=quick_url, inputs=q_url, outputs=[q_verdict,q_score])
            gr.Examples(
                examples=[
                    ["http://paypa1-secure.xyz/login?user=@victim"],
                    ["https://incometaxindia.gov.in"],
                    ["https://www.google.com"],
                    ["http://netflix-billing-update.net/confirm"],
                    ["https://iitb.ac.in/admissions"],
                    ["http://irs-refund-claim.xyz/verify"],
                ],
                inputs=q_url
            )

        with gr.TabItem("Try examples"):
            gr.Markdown("### Pre-loaded test cases — click any to run")
            gr.Examples(
                examples=[
                    ["URGENT: Your Netflix account is suspended. Verify now.", "http://netflix-billing-secure.xyz/login", None, "Netflix", False, False],
                    ["Your tax refund of $450.70 is ready. Claim within 24 hours.", "http://irs-refund-claim.net/verify?id=abc", None, "", True, True],
                    ["Hi, are we still meeting for coffee at 3pm?", "https://www.google.com/maps", None, "", False, False],
                    ["Dear customer, your SBI account needs reverification.", "http://sbi-account-verify.xyz/login", None, "SBI", False, False],
                    ["Your PAN card update is required. Click here.", "http://incometax-pan-update.net/verify", None, "", False, False],
                    ["Welcome to IRCTC. Your booking is confirmed.", "https://www.irctc.co.in/nget/train-search", None, "", False, False],
                ],
                inputs=[email_in, url_in, image_in, brand_in, spf_cb, dkim_cb],
                label="Click any row to load it, then click 'Run full threat scan'"
            )

demo.launch(share=True)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

/tmp/ipykernel_9009/2311588240.py:246: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=css, title="PhishGuard AI") as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2a3a6576208e061f8a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
